# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [ ]:
from dataclasses import dataclass
import csv
from pathlib import Path


from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

: 

## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [ ]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
rows = []
row = {}
with DATA_PATH.open(newline="", encoding="utf-8") as archivo:
    reader = csv.DictReader(archivo)
    for fila in reader:
        rows.append(fila)

row = rows[0]


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? Id
2. ¿Cuántas características numéricas hay? 11
3. ¿Qué columna extra añadirías para probar el contrato? Una vacia para ver si da error por tener más columnas de las esperadas.
4. ¿Qué valor de `ph` sería claramente inválido? Los que se salgan del rango, 200 por ejemplo

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo | Unidad |
|---|---|
| `fixed_acidity` | 0–20 | g/l |
| `volatile_acidity` | 0–2 | g/l |
| `citric_acid` | 0–2 | g/l |
| `residual_sugar` | 0–20 | g/l |
| `chlorides` | 0–1 | g/l |
| `free_sulfur_dioxide` | 0–100 | mg/l |
| `total_sulfur_dioxide` | 0–300 | mg/l |
| `density` | 0.98–1.01 | mg/cm^3 |
| `ph` | 2.5–4.5 | |
| `sulphates` | 0–3 | g/l |
| `alcohol` | 5–20 | % |

In [ ]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False,)
    
    # TODO: declara las 11 características con Field(ge=..., le=...).
  
    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field (ge=0, le=20)
    chlorides: float = Field (ge=0, le=1)
    free_sulfur_dioxide: float = Field (ge=0, le=100)
    total_sulfur_dioxide: float = Field (ge=0, le=300)
    density: float= Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)
    pass

# TODO: separa sample_id de la fila y valida solo las características del modelo.
sample_id = row["sample_id"]
model_row = {key: value for key, value in row.items() if key != "sample_id"}
request = WineQualityRequestPydantic.model_validate(model_row)


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas. Error de validacion

In [ ]:
# TODO: completa los casos y captura ValidationError.
casos = {
    "fila válida": model_row,
    "campo extra": {**model_row, "unexpected_field": 1},
    "ph fuera de rango": {**model_row, "ph": 99},
}
# try:
#     ...
# except ValidationError as error:
#     print(error)
for nombre, datos in casos.items():
    print(f"\n--- {nombre} ---")
    try:
        request = WineQualityRequestPydantic.model_validate(datos)
        print("Validación correcta:", request)
    except ValidationError as error:
        print("Validación fallida:")
        print(error)
sin_alcohol = model_row.copy()
sin_alcohol.pop("alcohol")

casos_ausentes = {
    "campo ausente": sin_alcohol,
    "celda vacía": {**model_row, "alcohol": ""},
    "valor None": {**model_row, "alcohol": None},
    "valor NaN": {**model_row, "alcohol": float("nan")},
}

for nombre, datos in casos_ausentes.items():
    try:
        WineQualityRequestPydantic.model_validate(datos)
    except ValidationError:
        print(f"OK: se rechaza {nombre}")
    else:
        raise AssertionError(f"No se ha rechazado: {nombre}")

NameError: name 'model_row' is not defined

## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.
request_ok = WineQualityRequestDataclass(
    fixed_acidity=7,
    volatile_acidity=1,
    citric_acid=0,
    residual_sugar=2,
    chlorides=1,
    free_sulfur_dioxide=11,
    total_sulfur_dioxide=34,
    density=1,
    ph=3,
    sulphates=2,
    alcohol=5,
)

print(request_ok)

### Decisión de diseño

Completa la frase:

> Usaría Pydantic en __________ porque __________. Usaría `dataclass` en __________ porque __________.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [ ]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
       "fixed_acidity",
        "volatile_acidity",
        "citric_acid",
        "residual_sugar",
        "chlorides",
        "free_sulfur_dioxide",
        "total_sulfur_dioxide",
        "density",
        "ph",
        "sulphates",
        "alcohol",
)

# TODO: construye vector usando getattr(request, name).

request = WineQualityRequestPydantic.model_validate(model_row)
vector = [getattr(request, name) for name in FEATURE_NAMES]

print(vector)
print("Número de características:", len(vector))

### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [ ]:
from typing import Literal
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
   model_config = ConfigDict(extra="forbid")
    
   sample_id: str = Field(min_length=1)
   quality_band: Literal["needs_review", "acceptable", "excellent"]
   confidence: float = Field(ge=0, le=1)
   model_version: str = Field(min_length=1)
   preprocessing_version: str = Field(min_length=1)

# TODO: escribe un ejemplo de salida para red-001.

prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="acceptable",
    confidence=0.85,
    model_version="modelo-ejemplo-v1",
    preprocessing_version="preprocesado-ejemplo-v1",
)

print(prediction.model_dump())

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

### 3 criterios comprobaciones

1. Entrada válida:
   Al procesar el CSV de cinco muestras, se generan cinco resultados
   con sample_id, quality_band, confidence, model_version y
   preprocessing_version. La confianza está entre 0 y 1.

2. Entrada inválida:
   Si faltan valores de una característica, aparece una columna extra o valores que se salgan del rango, genera un error.

3. Orden incompatible:
   Si las variables vienen en un orden distinto al esperado.